# 18 — Validated Mars process distributions and Earth matching\n
This notebook uses the actual process registry and review queue from notebook 06. It creates process-specific Mars candidate distributions. It matches only processes with both validated Mars evidence and an explicit Earth proxy table. Missing Earth variables produce a recorded blocked status, never a fabricated score.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOTS=[Path('/kaggle/working'),Path('/kaggle/input')]
def find(name):
    for root in ROOTS:
        if root.exists():
            hits=list(root.rglob(name))
            if hits:return hits[0]
    raise FileNotFoundError(name)

queue=pd.read_parquet(find('mars_process_review_queue.parquet'))
registry=pd.read_csv(find('mars_literature_process_registry.csv'))
distributions=pd.read_parquet(find('mars_neutral_type_measurement_distributions.parquet'))
out=Path('/kaggle/working/mars_validated_process_matching');out.mkdir(exist_ok=True)

# These are the process IDs and evidence statuses produced by notebook 06.
screenable={'regolith_mantling_or_dust_cover','water_driven_mineral_alteration','near_surface_ice_or_ground_ice'}
rows=[]
for pid in sorted(queue.process_id.unique()):
    q=queue[queue.process_id==pid]
    reg=registry[registry.process_id==pid]
    candidate_count=len(q)
    status='candidate_distribution_created' if pid in screenable else 'blocked_missing_required_mars_evidence'
    rows.append({'process_id':pid,'review_candidates':candidate_count,'required_families':reg.required_families.iloc[0] if len(reg) else None,'earth_variables':reg.earth_variables.iloc[0] if len(reg) else None,'status':status})
    if pid in screenable:
        q[['process_id','region_id','neutral_type','centroid_latitude','centroid_longitude','screening_score','status']].to_parquet(out/f'{pid}_mars_candidates.parquet',index=False)
readiness=pd.DataFrame(rows)
readiness.to_csv(out/'process_readiness.csv',index=False)
display(readiness)

# Existing notebook 09 is the only Earth proxy table currently available.
earth_files=list(ROOTS[0].rglob('refined_earth_matches.parquet'))+list(ROOTS[1].rglob('refined_earth_matches.parquet'))
if earth_files:
    dust=pd.read_parquet(earth_files[0]).copy()
    dust['process_id']='regolith_mantling_or_dust_cover'
    dust.to_parquet(out/'regolith_mantling_or_dust_cover_earth_candidates.parquet',index=False)
    print('Dust/regolith Earth matching table linked from notebook 09.')
for pid in ['water_driven_mineral_alteration','near_surface_ice_or_ground_ice']:
    print(pid, 'BLOCKED: current Earth inputs do not contain the required mineral/ice proxy measurements.')
(readiness[readiness.status=='candidate_distribution_created']).to_json(out/'validated_process_summary.json',orient='records',indent=2)
